### function

In [ ]:
import numpy as np
from sklearn.manifold import Isomap
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import shortest_path
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import time

def compute_geodesic_distances_isomap(X, n_neighbors=15, n_components=10, sample_fraction=0.05):
    """
    Compute approximate geodesic distance matrix for high-dimensional data using Isomap.
    
    Parameters:
        X: (n_samples, n_features) data matrix
        n_neighbors: number of k-nearest neighbors
        n_components: Isomap embedding dimension (for acceleration)
        sample_fraction: subsampling ratio (0~1), use 0.05~0.1 if memory is limited
    
    Returns:
        geodesic_dist: (n_samples_sub, n_samples_sub) geodesic distance matrix
        sample_idx: original indices of the subsampled points
    """
    if sample_fraction < 1.0:
        n_samples = X.shape[0]
        n_sub = int(n_samples * sample_fraction)
        np.random.seed(42)
        sample_idx = np.random.choice(n_samples, n_sub, replace=False)
        X_sub = X[sample_idx, :]
        print(f"Subsampled to {n_sub} samples")
    else:
        X_sub = X
        sample_idx = np.arange(X.shape[0])
        n_sub = X_sub.shape[0]
    
    # Use Isomap to approximate geodesic distances
    # Isomap builds a k-NN graph internally, then computes shortest paths
    print("Building Isomap model...")
    start_time = time.time()
    
    isomap = Isomap(
        n_neighbors=n_neighbors,
        n_components=n_components,
        metric='euclidean',
        n_jobs=-1
    )
    embedding = isomap.fit_transform(X_sub)
    
    # distance_matrix_ attribute contains geodesic distances
    if hasattr(isomap, 'distance_matrix_'):
        geodesic_dist = isomap.distance_matrix_
    else:
        # Fallback: manually compute shortest paths from k-NN graph
        from sklearn.neighbors import kneighbors_graph
        knn_graph = kneighbors_graph(X_sub, n_neighbors=n_neighbors, mode='distance', include_self=False)
        geodesic_dist = shortest_path(csgraph=knn_graph, directed=False, method='D')
    
    elapsed = time.time() - start_time
    print(f"Geodesic distance computation completed in {elapsed:.2f} seconds")
    print(f"Distance matrix shape: {geodesic_dist.shape}")
    
    return geodesic_dist, sample_idx




In [11]:
import numpy as np
import scanpy as sc
import pandas as pd
sc_data_path = "/home/qyyuan/project/ST_CP/JE/simulation/MERFISH/adata_sc_mouse1sample1_mouse1_slice50_celltype.h5ad"


In [12]:
import scanpy as sc
import numpy as np
adata = sc.read_h5ad(sc_data_path)


In [13]:
# ----------------- Usage Example -----------------
# X is your data matrix (19979, 1605)
# Subsampling is recommended due to large sample size
geodesic_dist, sample_idx = compute_geodesic_distances_isomap(
    adata.X, 
    n_neighbors=20,
    n_components=254,
    sample_fraction=1
)

print(f"Geodesic distance - Mean: {geodesic_dist.mean():.4f}, Std: {geodesic_dist.std():.4f}")

Building Isomap model...
Geodesic distance computation completed in 11.71 seconds
Distance matrix shape: (4198, 4198)
Geodesic distance - Mean: 122.1630, Std: 45.3118


In [15]:
np.save('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/geodesic_dist_sub.npy',geodesic_dist ,allow_pickle=True)
